# Dados e corpus de frases

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Igorbraziel/juristcu-clustering-tsdae/blob/main/notebooks/01_data_corpus.ipynb)

- **Objetivo:** descrever o JurisTCU e montar o corpus de frases sem rótulo para o TSDAE.
- **Entradas:** arquivo `doc.csv` do JurisTCU, no Hugging Face.
- **Saídas:** corpus de frases no Drive e tabelas descritivas em `results/tables/`.
- **Requisitos:** não requer GPU.

In [ ]:
# Configuração do ambiente. Fora do Colab, usa o ambiente local criado por `uv sync`.
import os
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
BRANCH = "main"  # Trocar pelo nome de uma branch de trabalho para testá-la.
REPO_URL = "https://github.com/Igorbraziel/juristcu-clustering-tsdae.git"
REPO_DIR = "/content/juristcu-clustering-tsdae"


def run(command: str) -> None:
    """Executa um comando de shell e interrompe a célula em caso de erro."""
    subprocess.run(command, shell=True, check=True)


if IN_COLAB:
    # Clona o repositório, se necessário, e posiciona na versão remota da branch.
    run(f"test -d {REPO_DIR} || git clone -q {REPO_URL} {REPO_DIR}")
    run(f"git -C {REPO_DIR} fetch -q origin {BRANCH}")
    run(f"git -C {REPO_DIR} checkout -q -B {BRANCH} FETCH_HEAD")
    # O uv costuma vir no Colab. Instala apenas se faltar.
    run("command -v uv > /dev/null || pip install -q uv")
    # --system: instala no Python do Colab, que já traz o torch com CUDA.
    # --no-sources: ignora o índice de torch CPU do pyproject.toml.
    # --no-config: ignora configurações locais do uv.
    # Instalação não editável: disponível sem reiniciar o ambiente de execução.
    # Nunca usar `uv sync` nem requirements.txt no Colab.
    run(f"uv pip install --system --no-config --no-sources -q {REPO_DIR}")
    # Tabelas e figuras são gravadas em results/, a partir da raiz do repositório.
    os.chdir(REPO_DIR)
    # Após atualizar o repositório com mudanças em src/, reiniciar o ambiente de execução.
    from google.colab import drive

    drive.mount("/content/drive")

In [ ]:
import importlib
import json

import pandas as pd

# Atualiza o cache de importação para reconhecer o pacote recém-instalado.
importlib.invalidate_caches()

from juristcu import corpus, data, storage  # noqa: E402
from juristcu.config import (  # noqa: E402
    AREA_COLUMN,
    N_SENTENCES,
    SEED,
    TEXT_COLUMN,
    TOPIC_COLUMN,
)

storage.save_environment("01")

## 1. Carregamento

- Carrega o arquivo `doc.csv` diretamente. O visualizador do Hugging Face falha com este conjunto.
- São esperados 16.045 documentos.

In [ ]:
documents = data.load_documents()
documents.shape

In [ ]:
documents.head()

## 2. Análise exploratória

- Distribuição das áreas e dos temas, para avaliar o equilíbrio entre eles.
- Tamanho dos enunciados em palavras, com percentis.
- O percentil 99 em palavras orienta o limite de tokens na codificação (`ENCODE_MAX_SEQ_LENGTH`).

In [ ]:
area_distribution = data.label_distribution(documents, AREA_COLUMN)
area_distribution

In [ ]:
topic_distribution = data.label_distribution(documents, TOPIC_COLUMN)
topic_distribution

In [ ]:
statement_lengths = documents[TEXT_COLUMN].str.split().str.len().rename("words")
length_percentiles = statement_lengths.describe(percentiles=[0.5, 0.9, 0.95, 0.99]).to_frame()
length_percentiles

## 3. Corpus de frases sem rótulo

- Fonte: ENUNCIADO e EXCERTO, sem áreas, temas ou outros metadados.
- Segmentação com o Punkt do NLTK. Frases de 5 a 80 palavras, sem duplicatas.
- Amostra de 20 mil frases com semente fixa, a mesma para as 3 sementes do TSDAE.

In [ ]:
corpus.ensure_nltk_resources()
sentences = corpus.build_corpus(documents, n_sentences=N_SENTENCES, seed=SEED)
len(sentences)

## 4. Inspeção manual

- Amostra aleatória de frases, para detectar ruído de segmentação.
- Tamanho das frases em palavras, para conferir o filtro de 5 a 80 palavras.

In [ ]:
pd.Series(sentences).sample(n=20, random_state=SEED).to_list()

In [ ]:
pd.Series(sentences).str.split().str.len().describe()

## 5. Salvamento

- Corpus de frases em `sentences.json`, no diretório de artefatos (Drive no Colab).
- Tabelas descritivas em `results/tables/`.

In [ ]:
sentences_path = storage.artifacts_dir() / "sentences.json"
sentences_path.write_text(json.dumps(sentences, ensure_ascii=False), encoding="utf-8")
sentences_path

In [ ]:
storage.save_table(area_distribution, "area_distribution")
storage.save_table(topic_distribution, "topic_distribution")
storage.save_table(length_percentiles.rename_axis("statistic").reset_index(), "statement_length")